# Sparkle V2 — Fase 3.6c: re-extracción con backbone mejorado

**Protocolo:** `Protocolo_Fase36_Camino_al_70.md` **Enmienda 3** (2026-07-29).

| | Actual (3.5a) | Este (3.6c) |
| :--- | :--- | :--- |
| Backbone | MobileNetV2 **α = 0,35** | MobileNetV2 **α = 1,0** |
| Entrada | **112 px** (pesos de 224, no nativos) | **128 px** (pesos nativos) |
| Pesos | ImageNet + fine-tuning FER2013 que casi no movió nada (§7.4) | **ImageNet puro** |
| Salida | 1.280-d | 1.280-d |
| Recorte, muestreo, `preprocess_input` | — | **idénticos** |

**Sin fine-tuning, a propósito**: aísla "¿el límite era la calidad del backbone?" de "¿el
límite era FER2013?".

**La intervención es un paquete** (α y resolución cambian juntos, porque α=1,0 a 112 px
volvería a cargar pesos no nativos). La conclusión se enuncia como *"rama de apariencia
mejorada vs. actual"*, nunca como *"α=1,0 le gana a α=0,35"* (regla 8).

**Costo estimado:** ~2-2,5 h en Colab T4. El cuello de botella sigue siendo la decodificación
de video y MediaPipe en CPU, no la CNN. Reanudable.

**El Test no se toca.**

In [ ]:
%pip install -q mediapipe kagglehub

In [ ]:
import sys, os, glob, json, time, shutil
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR, WORK_DIR = '/content/drive/MyDrive/SparkleV2', '/content'
else:
    PROJECT_DIR = os.environ.get('SPARKLE_PROJECT_DIR', './SparkleV2'); WORK_DIR = '.'
assert os.path.isdir(PROJECT_DIR), f'No existe {PROJECT_DIR}'

LABELS_DIR = f'{PROJECT_DIR}/labels'
RAW_DIR    = f'{PROJECT_DIR}/embeddings_raw_v2'       # destino nuevo (Drive)
RAW_LOCAL  = f'{WORK_DIR}/embeddings_raw_v2'
CACHE_32   = f'{PROJECT_DIR}/dataset_final_fase3/dataset_fase3_2_alineado.npz'
CACHE_V2   = f'{PROJECT_DIR}/dataset_final_fase3/dataset_fase3_6_raw_v2.npz'
CKPT_DIR   = f'{PROJECT_DIR}/checkpoints_fase3_6'
for s in ('Train', 'Validation', 'Test'):
    os.makedirs(f'{RAW_DIR}/{s}', exist_ok=True); os.makedirs(f'{RAW_LOCAL}/{s}', exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

import numpy as np, pandas as pd, cv2
import tensorflow as tf, keras
from keras import layers
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
from tqdm.auto import tqdm

gpus = tf.config.list_physical_devices('GPU')
print(f'TF {tf.__version__} | Keras {keras.__version__} | GPU: {gpus}')
if not gpus:
    print('\n*** SIN GPU: Entorno -> Cambiar tipo de entorno -> T4 GPU ***\n')

SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)

# ------- CONGELADO por la Enmienda 3 -------
CROP_SIZE    = 128        # resolución NATIVA de MobileNetV2 (96/128/160/192/224)
ALPHA        = 1.0
N_EMB_FRAMES = 16         # idéntico a 3.2a/3.5a
DIM_RAW      = 1280
D_PROJ       = 16         # el d congelado en §11.7
PROJ_SEED    = 42         # semilla de la proyección, declarada de antemano
DTYPE_RAW    = np.float16
SYNC_EVERY   = 200

def setup_kaggle_credentials():
    if os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY'):
        return 'ya configuradas'
    if IN_COLAB:
        try:
            from google.colab import userdata
            os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
            os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
            return 'Secretos de Colab'
        except Exception:
            pass
    for c in (f'{PROJECT_DIR}/kaggle.json', '/content/drive/MyDrive/kaggle.json',
              os.path.expanduser('~/.kaggle/kaggle.json')):
        if os.path.exists(c):
            d = json.load(open(c))
            os.environ['KAGGLE_USERNAME'] = d['username']; os.environ['KAGGLE_KEY'] = d['key']
            return f'kaggle.json en {c}'
    if IN_COLAB:
        from google.colab import files
        print('No encontré credenciales. Subí tu kaggle.json:')
        up = files.upload(); nombre = list(up)[0]
        d = json.loads(up[nombre])
        os.environ['KAGGLE_USERNAME'] = d['username']; os.environ['KAGGLE_KEY'] = d['key']
        # Se persiste en Drive para que ningún notebook vuelva a pedirlo
        try:
            with open(f'{PROJECT_DIR}/kaggle.json', 'w') as f:
                json.dump(d, f)
            return 'subida manual (guardada en Drive para próximas corridas)'
        except Exception:
            return 'subida manual'
    raise RuntimeError('Sin credenciales de Kaggle')

print('Credenciales:', setup_kaggle_credentials())

## 1. DAiSEE, etiquetas e índice de videos

In [ ]:
import kagglehub
def find_dataset_root(base):
    for root, dirs, _ in os.walk(base):
        if {'Train', 'Validation', 'Test'}.issubset(set(dirs)):
            return root
    return base

t0 = time.time()
RAW_DATASET_DIR = find_dataset_root(kagglehub.dataset_download('olgaparfenova/daisee'))
print(f'DAiSEE en {RAW_DATASET_DIR} ({(time.time()-t0)/60:.1f} min)')

labels = {}
for s in ('Train', 'Validation', 'Test'):
    df = pd.read_csv(f'{LABELS_DIR}/{s}Labels.csv'); df.columns = [c.strip() for c in df.columns]
    labels[s] = df
video_index = {}
for s in ('Train', 'Validation', 'Test'):
    for ext in ('avi', 'mp4'):
        for p in glob.glob(f'{RAW_DATASET_DIR}/{s}/**/*.{ext}', recursive=True):
            video_index[os.path.basename(p)] = p
print(f'Videos indexados: {len(video_index)}')
assert video_index

## 2. Recorte y muestreo — funciones idénticas a 3.2a/3.5a

Lo único que cambia es `CROP_SIZE`. El bounding box, el margen del 25 %, el recorte cuadrado y
el muestreo de 16 frames son literalmente los mismos.

In [ ]:
import urllib.request
MODEL_PATH = f'{WORK_DIR}/face_landmarker.task'
if not os.path.exists(MODEL_PATH):
    urllib.request.urlretrieve(
        'https://storage.googleapis.com/mediapipe-models/face_landmarker/'
        'face_landmarker/float16/latest/face_landmarker.task', MODEL_PATH)

def create_face_landmarker():
    return mp_vision.FaceLandmarker.create_from_options(mp_vision.FaceLandmarkerOptions(
        base_options=mp_python.BaseOptions(model_asset_path=MODEL_PATH),
        running_mode=mp_vision.RunningMode.IMAGE,
        num_faces=1, min_face_detection_confidence=0.5))

def crop_face_from_landmarks(frame_bgr, landmarks, crop_size=CROP_SIZE, margin=0.25):
    h, w = frame_bgr.shape[:2]
    xs = np.array([lm.x for lm in landmarks]) * w
    ys = np.array([lm.y for lm in landmarks]) * h
    x_min, x_max, y_min, y_max = xs.min(), xs.max(), ys.min(), ys.max()
    side = max(x_max - x_min, y_max - y_min) * (1 + margin)
    cx, cy = (x_min + x_max) / 2, (y_min + y_max) / 2
    x0, y0 = int(max(0, cx - side/2)), int(max(0, cy - side/2))
    x1, y1 = int(min(w, cx + side/2)), int(min(h, cy + side/2))
    if x1 <= x0 or y1 <= y0:
        return None
    return cv2.cvtColor(cv2.resize(frame_bgr[y0:y1, x0:x1], (crop_size, crop_size),
                                   interpolation=cv2.INTER_AREA), cv2.COLOR_BGR2RGB)

def extract_crop_from_frame(detector, frame_bgr):
    res = detector.detect(mp.Image(image_format=mp.ImageFormat.SRGB,
                                   data=cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)))
    return crop_face_from_landmarks(frame_bgr, res.face_landmarks[0]) if res.face_landmarks else None

def sample_frames_sequential(video_path, n_frames=N_EMB_FRAMES):
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release(); return None
    wanted = np.linspace(0, total - 1, n_frames).astype(int)
    ws = {}
    for pos, idx in enumerate(wanted):
        ws.setdefault(int(idx), []).append(pos)
    frames = [None]*n_frames; i, mx = 0, int(wanted.max())
    while i <= mx:
        ret, fr = cap.read()
        if not ret:
            break
        if i in ws:
            for pos in ws[i]:
                frames[pos] = fr
        i += 1
    cap.release(); return frames

## 3. Backbone nuevo — α = 1,0 a 128 px, ImageNet, sin fine-tuning

A 128 px los pesos preentrenados son **nativos**: no debería aparecer el warning
`rows is not in [96,128,160,192,224]` que sí salía a 112 px.

In [ ]:
backbone = keras.applications.MobileNetV2(
    input_shape=(CROP_SIZE, CROP_SIZE, 3), alpha=ALPHA,
    include_top=False, weights='imagenet', pooling='avg')
backbone.trainable = False
assert int(backbone.output_shape[-1]) == DIM_RAW, backbone.output_shape

inp = keras.Input((CROP_SIZE, CROP_SIZE, 3), name='crop')
extractor = keras.Model(inp, backbone(inp, training=False), name='cnn_raw_v2')
extractor.trainable = False
print(f'Backbone α={ALPHA} @{CROP_SIZE}: {backbone.count_params():,} parámetros, '
      f'salida {DIM_RAW}-d')

# Proyección congelada (Enmienda 3): una realización, semilla declarada.
rng = np.random.default_rng(PROJ_SEED)
W_v2 = rng.normal(0, 1/np.sqrt(DIM_RAW), size=(DIM_RAW, D_PROJ)).astype(np.float32)
b_v2 = np.zeros(D_PROJ, dtype=np.float32)
# Y 5 realizaciones extra, SOLO para el control de robustez de 3.6d
W_alt = np.stack([np.random.default_rng(1000+s).normal(
    0, 1/np.sqrt(DIM_RAW), size=(DIM_RAW, D_PROJ)).astype(np.float32) for s in range(5)])
np.savez(f'{CKPT_DIR}/proyeccion_v2.npz', W=W_v2, b=b_v2, W_alt=W_alt,
         proj_seed=PROJ_SEED, d=D_PROJ, alpha=ALPHA, crop=CROP_SIZE)
print(f'Proyección {W_v2.shape} guardada (semilla {PROJ_SEED}) + 5 alternativas de control')

def process_clip_raw(video_path, detector, extractor, n_frames=N_EMB_FRAMES):
    frames = sample_frames_sequential(video_path, n_frames)
    if frames is None:
        return None
    crops = np.zeros((n_frames, CROP_SIZE, CROP_SIZE, 3), dtype=np.float32)
    valid = np.zeros(n_frames, dtype=bool)
    for pos, fr in enumerate(frames):
        if fr is None:
            continue
        c = extract_crop_from_frame(detector, fr)
        if c is not None:
            crops[pos] = keras.applications.mobilenet_v2.preprocess_input(c.astype(np.float32))
            valid[pos] = True
    if not valid.any():
        return np.full((n_frames, DIM_RAW), np.nan, dtype=np.float32)
    out = np.asarray(extractor(crops, training=False)).astype(np.float32)
    out[~valid] = np.nan
    return out

# --- Prueba de humo: 5 clips, verificar velocidad y que los embeddings no sean degenerados ---
det = create_face_landmarker()
try:
    t0 = time.time(); normas = []
    for cid in labels['Train']['ClipID'].head(5):
        if cid not in video_index:
            continue
        e = process_clip_raw(video_index[cid], det, extractor)
        normas.append(float(np.nanmean(np.linalg.norm(np.nan_to_num(e), axis=1))))
        print(f'{cid}: {np.isnan(e).any(axis=1).mean()*100:5.1f}% sin rostro | norma {normas[-1]:.2f}')
    dt = (time.time()-t0)/max(len(normas), 1)
finally:
    det.close()
print(f'\n{dt:.2f} s/clip → estimado {dt*8571/3600:.1f} h para 8.571 clips')
print('(3.5a fue 0,69 s/clip = 1 h 45)')

## 4. Extracción completa — reanudable

In [ ]:
def sync_to_drive(split):
    src, dst = f'{RAW_LOCAL}/{split}', f'{RAW_DIR}/{split}'
    n = 0
    for f in os.listdir(src):
        if not os.path.exists(f'{dst}/{f}'):
            shutil.copy2(f'{src}/{f}', f'{dst}/{f}'); n += 1
    return n

def already_done(split):
    done = set()
    for base in (f'{RAW_DIR}/{split}', f'{RAW_LOCAL}/{split}'):
        if os.path.isdir(base):
            done |= {os.path.splitext(f)[0] for f in os.listdir(base) if f.endswith('.npz')}
    return done

for SPLIT in ['Train', 'Validation', 'Test']:
    out_dir = f'{RAW_LOCAL}/{SPLIT}'; log_path = f'{RAW_DIR}/{SPLIT}_log.csv'
    log = pd.read_csv(log_path).to_dict('records') if os.path.exists(log_path) else []
    done = already_done(SPLIT)
    pend = [r for _, r in labels[SPLIT].iterrows()
            if os.path.splitext(r['ClipID'])[0] not in done]
    print(f'\n=== {SPLIT}: {len(done)} hechos, {len(pend)} pendientes ===')
    if not pend:
        sync_to_drive(SPLIT); continue
    det = create_face_landmarker(); n_ok = 0
    try:
        for row in tqdm(pend, desc=SPLIT):
            cid = row['ClipID']; key = os.path.splitext(cid)[0]
            if cid not in video_index:
                log.append({'ClipID': cid, 'status': 'sin_video_fisico'}); continue
            try:
                e = process_clip_raw(video_index[cid], det, extractor)
                if e is None:
                    log.append({'ClipID': cid, 'status': 'video_ilegible'}); continue
                np.savez_compressed(f'{out_dir}/{key}.npz', embeddings=e.astype(DTYPE_RAW))
                log.append({'ClipID': cid, 'status': 'ok',
                            'pct_nan': float(np.isnan(e).any(axis=1).mean()*100)})
                n_ok += 1
            except Exception as ex:
                log.append({'ClipID': cid, 'status': f'error: {type(ex).__name__}: {ex}'})
            if n_ok and n_ok % SYNC_EVERY == 0:
                n = sync_to_drive(SPLIT); pd.DataFrame(log).to_csv(log_path, index=False)
                tqdm.write(f'  [checkpoint] {n_ok} clips | {n} nuevos a Drive')
    finally:
        det.close(); n = sync_to_drive(SPLIT)
        pd.DataFrame(log).to_csv(log_path, index=False)
        print(f'{SPLIT}: {n} archivos sincronizados')

for SPLIT in ['Train', 'Validation', 'Test']:
    n = len(glob.glob(f'{RAW_DIR}/{SPLIT}/*.npz'))
    lp = f'{RAW_DIR}/{SPLIT}_log.csv'
    st = pd.read_csv(lp)['status'].astype(str).str.split(':').str[0].value_counts().to_dict() \
         if os.path.exists(lp) else {}
    print(f'{SPLIT}: {n} .npz | {st}')

## 5. Caché consolidada — alineada al MISMO orden de ClipIDs

Se recorre el orden de `dataset_fase3_2_alineado.npz`, igual que en 3.5a §7, para que la
geometría, las etiquetas, los IDs y los folds sean bit a bit los mismos. Lo único distinto
entre esta corrida y la de §13 es la rama de apariencia.

In [ ]:
d32 = np.load(CACHE_32, allow_pickle=True)
def apilar(split, ids):
    ids = [str(c) for c in ids]
    faltan = [c for c in ids if not os.path.exists(f'{RAW_DIR}/{split}/{c}.npz')]
    assert not faltan, f'{len(faltan)} clips sin extraer en {split} (ej. {faltan[:3]})'
    out = np.empty((len(ids), N_EMB_FRAMES, DIM_RAW), dtype=DTYPE_RAW)
    for i, c in enumerate(tqdm(ids, desc=f'consolidando {split}')):
        out[i] = np.load(f'{RAW_DIR}/{split}/{c}.npz')['embeddings']
    return out

Xv_tr = apilar('Train', d32['id_tr'])
Xv_va = apilar('Validation', d32['id_va'])
Xv_te = apilar('Test', d32['id_te'])
np.savez(CACHE_V2, Xr_tr=Xv_tr, Xr_va=Xv_va, Xr_te=Xv_te,
         id_tr=d32['id_tr'], id_va=d32['id_va'], id_te=d32['id_te'],
         dim_raw=DIM_RAW, alpha=ALPHA, crop=CROP_SIZE)
print(f'\n{CACHE_V2} ({os.path.getsize(CACHE_V2)/1e6:.0f} MB)')
print(f'Train {Xv_tr.shape} | Validation {Xv_va.shape} | Test {Xv_te.shape}')

## 6. Control de calidad — comparable con §10.4

In [ ]:
M = Xv_tr[:300].astype(np.float32)
cm = np.nanmean(M, axis=1)
inter = float(np.nanstd(cm, axis=0).mean()); intra = float(np.nanstd(M, axis=1).mean())
V = np.nan_to_num(cm); V = V/(np.linalg.norm(V, axis=1, keepdims=True)+1e-9)
cos = float((V @ V.T)[np.triu_indices(len(V), k=1)].mean())
print(f'Frames sin rostro      : {np.isnan(M).any(axis=2).mean()*100:.2f} %')
print(f'Norma media por clip   : {np.linalg.norm(V*0 + np.nan_to_num(cm), axis=1).mean():.3f}')
print(f'Desvío ENTRE clips     : {inter:.4f}')
print(f'Desvío DENTRO del clip : {intra:.4f}')
print(f'Similitud coseno media : {cos:.3f}  (alarma si > 0,98)')
print(f'Relación entre/dentro  : {inter/max(intra,1e-9):.2f}   (3.2a: 2,05 · 3.5a: 2,42)')
assert cos < 0.98, 'Embeddings degenerados'

## 7. Checkpoint

Antes de pasar a 3.6d, confirmar:

1. Los tres splits cerraron con 0 errores.
2. `dataset_fase3_6_raw_v2.npz` existe y sus shapes coinciden con las de 3.5a.
3. La similitud coseno quedó lejos de 0,98.
4. **No apareció** el warning `rows is not in [96,128,160,192,224]` — a 128 px los pesos son
   nativos. Si aparece, algo quedó en 112 px y hay que revisarlo antes de seguir.

Recordatorio del protocolo: la conclusión se enuncia como *"rama de apariencia mejorada vs.
actual"*, **nunca** como *"α=1,0 le gana a α=0,35"* — cambian dos cosas a la vez y así está
declarado (Enmienda 3, regla 8).

**Evaluaciones sobre Test consumidas: 0.**